# Q1 Data Request: Monthly Scheme Calendar + Pack-Level Revenue

This notebook uses parquet files from `DATA` and produces:

1. **Monthly scheme calendar (Pack Size x Slab Level)** using tool-style slab logic (recomputed, not file slab)
2. **Monthly pack-level metrics**:
   - Volume = `Quantity`
   - Gross Revenue = `Quantity ? DSP`
   - Net Revenue = `Quantity ? CLP`

## New Methodology Mapping
- CLP = `Selling_Rate_Without_GST_CLP`
- Quantity Including Free Issue = `Quantity`
- DSP = `Basic_Rate_Per_PC_without_GST` (fallback to `Basic_Rate_Per_PC`)
- Base Scheme = `Scheme_Discount`
- QPS on top of Base Scheme = `Staggered_qps`


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)


In [ ]:
# Paths
ROOT = Path.cwd().resolve().parents[1] if (Path.cwd().name == 'q1') else Path.cwd()
DATA_DIR = ROOT / 'DATA'
OUT_DIR = ROOT / 'data_requests' / 'q1' / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT    :', ROOT)
print('DATA_DIR:', DATA_DIR)
print('OUT_DIR :', OUT_DIR)


In [ ]:
# Scope / filters
TARGET_SUBCATEGORIES = {'STX INSTA SHAMPOO', 'STREAX INSTA SHAMPOO'}
TARGET_SIZES = {'12-ML', '18-ML'}
OUTLET_TYPE_FILTER = 'GT'      # set None to skip
STATE_FILTER = None            # example: {'MAH', 'UP'} or None


In [ ]:
# Canonical aliases
ALIASES = {
    'date': 'Date',
    'state': 'State',
    'outlet_type': 'Outlet_Type',
    'subcategory': 'Subcategory',
    'sizes': 'Sizes',
    'quantity': 'Quantity',
    'store_id': 'Store_ID',
    'distributor_id': 'Distributor_ID',
    'outlet_id': 'Outlet_ID',
    'scheme_discount': 'Scheme_Discount',
    'staggered_qps': 'Staggered_qps',
    'basic_rate_per_pc_without_gst': 'DSP',
    'basic_rate_per_pc': 'DSP',
    'selling_rate_without_gst_clp': 'CLP',
}

REQUIRED_BASE = ['Date', 'Subcategory', 'Sizes', 'Quantity', 'Scheme_Discount', 'Staggered_qps', 'DSP', 'CLP']
POTENTIAL_STORE = ['Store_ID', 'Distributor_ID', 'Outlet_ID']
OPTIONAL = ['Outlet_Type', 'State']

def norm_key(x: str) -> str:
    return re.sub(r'[^a-z0-9]+', '_', str(x).strip().lower()).strip('_')

def normalize_size(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.upper().str.replace(' ', '', regex=False).str.strip()
    return s.replace({'12': '12-ML', '12ML': '12-ML', '18': '18-ML', '18ML': '18-ML'})

def normalize_text(series: pd.Series) -> pd.Series:
    return series.astype(str).str.upper().str.replace(r'\s+', ' ', regex=True).str.strip()


In [ ]:
# Read and normalize parquet files
files = sorted(DATA_DIR.glob('*.parquet'))
if not files:
    raise FileNotFoundError(f'No parquet files found in {DATA_DIR}')

parts = []
for fp in files:
    schema_cols = pq.ParquetFile(fp).schema.names
    canonical_to_source = {}
    for src in schema_cols:
        k = norm_key(src)
        canon = ALIASES.get(k)
        if canon and canon not in canonical_to_source:
            canonical_to_source[canon] = src

    needed = list(dict.fromkeys(REQUIRED_BASE + POTENTIAL_STORE + OPTIONAL))
    available = [c for c in needed if c in canonical_to_source]
    missing_required = [c for c in REQUIRED_BASE if c not in canonical_to_source]
    if missing_required:
        raise ValueError(f'{fp.name} missing required columns: {missing_required}')

    read_cols = [canonical_to_source[c] for c in available]
    df = pd.read_parquet(fp, columns=read_cols)
    rename_map = {canonical_to_source[c]: c for c in available}
    df = df.rename(columns=rename_map)
    df['__source_file'] = fp.name
    parts.append(df)

raw = pd.concat(parts, ignore_index=True)
print('Loaded rows:', f"{len(raw):,}")
print('Columns:', sorted(raw.columns.tolist()))


In [ ]:
# Type normalization + filters
work = raw.copy()
work['Date'] = pd.to_datetime(work['Date'], errors='coerce')
for c in ['Quantity', 'Scheme_Discount', 'Staggered_qps', 'DSP', 'CLP']:
    work[c] = pd.to_numeric(work[c], errors='coerce')

work['Subcategory'] = normalize_text(work['Subcategory'])
work['Sizes'] = normalize_size(work['Sizes'])

mask = (
    work['Date'].notna()
    & work['Quantity'].notna()
    & (work['Quantity'] > 0)
    & work['Subcategory'].isin(TARGET_SUBCATEGORIES)
    & work['Sizes'].isin(TARGET_SIZES)
)

if OUTLET_TYPE_FILTER is not None and 'Outlet_Type' in work.columns:
    work['Outlet_Type'] = normalize_text(work['Outlet_Type'])
    mask &= work['Outlet_Type'].eq(str(OUTLET_TYPE_FILTER).upper())

if STATE_FILTER is not None and 'State' in work.columns:
    state_set = {str(s).upper().strip() for s in STATE_FILTER}
    work['State'] = normalize_text(work['State'])
    mask &= work['State'].isin(state_set)

f = work.loc[mask].copy()
print('Filtered rows:', f"{len(f):,}")
print('Month range:', f['Date'].min(), 'to', f['Date'].max())
print('Sizes:', sorted(f['Sizes'].dropna().unique().tolist()))
print('Subcategories:', sorted(f['Subcategory'].dropna().unique().tolist()))


In [ ]:
# Build Store_ID for slab logic
if 'Store_ID' in f.columns:
    sid = f['Store_ID'].astype(str).str.strip()
    has_sid = sid.ne('') & sid.ne('nan')
else:
    has_sid = pd.Series(False, index=f.index)

if has_sid.any():
    f['Store_Key'] = sid
else:
    if not {'Distributor_ID', 'Outlet_ID'}.issubset(f.columns):
        raise ValueError('Need either Store_ID or both Distributor_ID and Outlet_ID for slab assignment.')
    f['Store_Key'] = f['Distributor_ID'].astype(str).str.strip() + ' ' + f['Outlet_ID'].astype(str).str.strip()

f['Month'] = f['Date'].dt.to_period('M').astype(str)

monthly_outlet_qty = (
    f.groupby(['Month', 'Sizes', 'Store_Key'], as_index=False)
     .agg(monthly_outlet_qty=('Quantity', 'sum'))
)

monthly_outlet_qty.head()


In [ ]:
# Tool-style slab assignment (recomputed)
def assign_slab(size: str, q: float) -> str:
    if size == '12-ML':
        # slab0 <8, slab1 8-<144, slab2 >=144
        if q < 8:
            return 'slab0'
        if q < 144:
            return 'slab1'
        return 'slab2'
    if size == '18-ML':
        # slab0 <8, slab1 8-<32, slab2 32-<576, slab3 576-<960, slab4 >=960
        if q < 8:
            return 'slab0'
        if q < 32:
            return 'slab1'
        if q < 576:
            return 'slab2'
        if q < 960:
            return 'slab3'
        return 'slab4'
    return 'slab0'

monthly_outlet_qty['Slab'] = monthly_outlet_qty.apply(
    lambda r: assign_slab(r['Sizes'], r['monthly_outlet_qty']),
    axis=1
)

# Map slab back to transactional rows
slab_map = monthly_outlet_qty[['Month', 'Sizes', 'Store_Key', 'Slab']]
f = f.merge(slab_map, on=['Month', 'Sizes', 'Store_Key'], how='left')

print('Rows with slab assigned:', f['Slab'].notna().sum(), '/', len(f))
f[['Month', 'Sizes', 'Store_Key', 'Slab']].head()


In [ ]:
# Derived metrics
f['Base_Scheme'] = f['Scheme_Discount'].fillna(0.0)
f['QPS_Topup'] = f['Staggered_qps'].fillna(0.0)
f['Total_Scheme'] = f['Base_Scheme'] + f['QPS_Topup']

f['Gross_Revenue'] = f['Quantity'] * f['DSP']
f['Net_Revenue'] = f['Quantity'] * f['CLP']

f[['Quantity', 'DSP', 'CLP', 'Gross_Revenue', 'Net_Revenue']].describe().T


In [ ]:
# Output 1: Monthly Scheme Calendar (Pack x Slab)
scheme_calendar = (
    f.groupby(['Month', 'Sizes', 'Slab'], as_index=False)
     .apply(lambda g: pd.Series({
         'Volume_Qty': g['Quantity'].sum(),
         'Base_Scheme_%': np.average(g['Base_Scheme'], weights=g['Quantity']) if g['Quantity'].sum() > 0 else np.nan,
         'QPS_Topup_%': np.average(g['QPS_Topup'], weights=g['Quantity']) if g['Quantity'].sum() > 0 else np.nan,
         'Total_Scheme_%': np.average(g['Total_Scheme'], weights=g['Quantity']) if g['Quantity'].sum() > 0 else np.nan,
     }))
     .reset_index(drop=True)
)

scheme_calendar = scheme_calendar.sort_values(['Month', 'Sizes', 'Slab']).reset_index(drop=True)
scheme_calendar.head(20)


In [ ]:
# Output 2: Monthly pack-level Volume / Gross Revenue / Net Revenue
pack_monthly = (
    f.groupby(['Month', 'Sizes'], as_index=False)
     .agg(
         Volume_Qty=('Quantity', 'sum'),
         Gross_Revenue=('Gross_Revenue', 'sum'),
         Net_Revenue=('Net_Revenue', 'sum')
     )
     .sort_values(['Month', 'Sizes'])
     .reset_index(drop=True)
)

pack_monthly.head(20)


In [ ]:
# Optional manager views (pivot)
calendar_12 = scheme_calendar[scheme_calendar['Sizes'] == '12-ML'].pivot(index='Month', columns='Slab', values='Total_Scheme_%').sort_index()
calendar_18 = scheme_calendar[scheme_calendar['Sizes'] == '18-ML'].pivot(index='Month', columns='Slab', values='Total_Scheme_%').sort_index()

print('12-ML Scheme Calendar (Total Scheme %):')
display(calendar_12)
print('18-ML Scheme Calendar (Total Scheme %):')
display(calendar_18)


In [ ]:
# Save outputs
scheme_csv = OUT_DIR / 'monthly_scheme_calendar_pack_slab.csv'
pack_csv = OUT_DIR / 'monthly_pack_revenue_volume.csv'
excel_out = OUT_DIR / 'q1_monthly_scheme_and_pack_outputs.xlsx'

scheme_calendar.to_csv(scheme_csv, index=False)
pack_monthly.to_csv(pack_csv, index=False)

with pd.ExcelWriter(excel_out, engine='openpyxl') as xw:
    scheme_calendar.to_excel(xw, sheet_name='scheme_calendar_pack_slab', index=False)
    pack_monthly.to_excel(xw, sheet_name='pack_monthly_rev_vol', index=False)
    calendar_12.to_excel(xw, sheet_name='calendar_12ml_total_scheme')
    calendar_18.to_excel(xw, sheet_name='calendar_18ml_total_scheme')

print('Saved:')
print('-', scheme_csv)
print('-', pack_csv)
print('-', excel_out)
